# 🌿 Crop Disease Detection using CNN
**Course:** Artificial Intelligence — Semester 6
**Supervision:** Prof. Tajjamul
**Author:** Muhammad Taha Ahmad ([@NinjaVinja](https://github.com/NinjaVinja))

Trains a CNN on the PlantVillage dataset to detect crop diseases from leaf images (tomato, potato, bell pepper).

**Steps:**
1. Setup & imports
2. Download the dataset
3. Preprocess the data
4. Build the CNN
5. Baseline evaluation (before training)
6. Train the model
7. Evaluate + log results across runs
8. Save the model
9. Test on your own uploaded photo

> Go to **Runtime → Change runtime type → GPU** before running (free GPU on Colab, training is much faster).

## Step 1: Imports & Config

In [ ]:
import os
import subprocess
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 15
VALIDATION_SPLIT = 0.2
MODEL_OUT_PATH = "crop_disease_model.h5"
LOG_FILE = "training_log.csv"


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

## Step 2: Download the Dataset (PlantVillage via kagglehub)

If this fails (sometimes Kaggle login is required), see the manual download steps in the README.

In [ ]:
def ensure_kagglehub_installed():
    try:
        import kagglehub  # noqa: F401
    except ImportError:
        print("kagglehub not found — installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])


def get_dataset_dir() -> str:
    try:
        ensure_kagglehub_installed()
        import kagglehub
        path = kagglehub.dataset_download("emmarex/plantdisease")
        dataset_dir = os.path.join(path, "PlantVillage")
    except Exception as err:
        print(f"kagglehub download failed ({err}); falling back to local ./PlantVillage")
        dataset_dir = "PlantVillage"

    if not os.path.isdir(dataset_dir):
        raise FileNotFoundError(
            f"Could not find dataset at {dataset_dir}. "
            "Download it manually from Kaggle and place it in this folder — "
            "see the README's 'Manual dataset download' section."
        )
    return dataset_dir


dataset_dir = get_dataset_dir()
print("Dataset path:", dataset_dir)
print("Classes:", os.listdir(dataset_dir))

## Step 3: Data Preprocessing

- Resize all images to 128x128
- Normalize pixel values (0–1)
- Split into train/validation (80/20)
- Light augmentation (rotation, zoom, horizontal flip) for better generalization

In [ ]:
def build_data_generators(dataset_dir: str):
    datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        validation_split=VALIDATION_SPLIT,
        rotation_range=20,
        zoom_range=0.2,
        horizontal_flip=True,
    )

    train_data = datagen.flow_from_directory(
        dataset_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="training",
    )

    val_data = datagen.flow_from_directory(
        dataset_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="validation",
    )

    return train_data, val_data


train_data, val_data = build_data_generators(dataset_dir)

num_classes = train_data.num_classes
class_names = list(train_data.class_indices.keys())
print("Total classes (diseases):", num_classes)
print("Class names:", class_names)

## Step 4: Preview Sample Images (Sanity Check)

In [ ]:
def preview_samples(train_data, class_names, n=6):
    images, labels = next(train_data)
    plt.figure(figsize=(12, 8))
    for i in range(n):
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i])
        plt.title(class_names[np.argmax(labels[i])], fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


preview_samples(train_data, class_names)

## Step 5: Build the CNN Model

Three Conv+Pool blocks to pull out edges/color/texture features, then a dense head with dropout to reduce overfitting.

In [ ]:
def build_model(num_classes: int) -> Sequential:
    model = Sequential([
        Conv2D(32, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),

        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ])

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


model = build_model(num_classes)
model.summary()

## Step 5a: Baseline Result (Before Training)

The model hasn't learned anything yet — this is close to random guessing, so we know how much training actually helps.

In [ ]:
print("=== Baseline result (before training) ===")
baseline_loss, baseline_acc = model.evaluate(val_data)
print(f"Before training -> Loss: {baseline_loss:.4f} | Accuracy: {baseline_acc * 100:.2f}%")

## Step 6: Train the Model

With GPU, 15 epochs should take a few minutes.

In [ ]:
history = model.fit(train_data, validation_data=val_data, epochs=EPOCHS)

## Step 6a: Result After Training

In [ ]:
print("=== Result (after training) ===")
final_loss, final_acc = model.evaluate(val_data)
print(f"After training -> Loss: {final_loss:.4f} | Accuracy: {final_acc * 100:.2f}%")
print(f"Improvement over baseline: {(final_acc - baseline_acc) * 100:.2f}%")

## Step 6b: Log This Run + View All Previous Runs

Every run's numbers get appended to `training_log.csv` instead of being overwritten, so multiple experiments are comparable.

In [ ]:
def log_run(baseline_acc, final_train_acc, final_val_acc, final_val_loss):
    record = {
        "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "epochs": EPOCHS,
        "baseline_accuracy_%": round(baseline_acc * 100, 2),
        "final_train_accuracy_%": round(final_train_acc * 100, 2),
        "final_val_accuracy_%": round(final_val_acc * 100, 2),
        "final_val_loss": round(final_val_loss, 4),
    }

    if os.path.exists(LOG_FILE):
        log_df = pd.read_csv(LOG_FILE)
        log_df = pd.concat([log_df, pd.DataFrame([record])], ignore_index=True)
    else:
        log_df = pd.DataFrame([record])

    log_df.to_csv(LOG_FILE, index=False)
    return log_df


log_df = log_run(baseline_acc, history.history["accuracy"][-1], final_acc, final_loss)
print(f"Logged this run. Total runs so far: {len(log_df)}")
log_df

## Step 6c: Comparison Graph of All Training Runs

In [ ]:
def plot_run_history(log_df):
    plt.figure(figsize=(10, 5))
    run_numbers = range(1, len(log_df) + 1)
    plt.plot(run_numbers, log_df["baseline_accuracy_%"], marker="x", linestyle="--", label="Baseline (before training)")
    plt.plot(run_numbers, log_df["final_val_accuracy_%"], marker="o", label="Validation accuracy (after training)")
    plt.xlabel("Training run #")
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy across all training runs")
    plt.xticks(list(run_numbers))
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


plot_run_history(log_df)

## Step 7: Training Curves (Accuracy & Loss per Epoch)

In [ ]:
def plot_training_curves(history):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Train accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation accuracy")
    plt.title("Accuracy over epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Train loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.title("Loss over epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_training_curves(history)

## Step 8: Save the Model

In [ ]:
model.save(MODEL_OUT_PATH)
print(f"Model saved to {MODEL_OUT_PATH}")

## Step 9: Test on Your Own Leaf Photo

Opens a file-upload picker (Colab only) so you can test the model on a real photo.

In [ ]:
def predict_image(model, image_path, class_names):
    from tensorflow.keras.preprocessing import image as keras_image

    img = keras_image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%")
    plt.show()

    return predicted_class, confidence


def predict_uploaded_image(model, class_names):
    if not in_colab():
        print("This only works inside Google Colab.")
        print("Locally, call predict_image(model, 'path/to/image.jpg', class_names) instead.")
        return

    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded.keys():
        predict_image(model, filename, class_names)


predict_uploaded_image(model, class_names)

## 🎯 Summary

1. **Problem:** Detect crop disease from leaf images
2. **Dataset:** PlantVillage (Kaggle) — tomato/potato/pepper leaves, healthy + diseased
3. **Model:** CNN — 3 Conv layers + Dense layers
4. **Learning type:** Supervised (every image has a known label)
5. **Training:** via `model.fit()`
6. **Result:** Baseline vs. final accuracy comparison, plus a run history log
7. **Real-world impact:** A farmer could photograph a leaf and instantly identify the disease, helping prevent crop loss

**Next step:** Transfer learning (pretrained MobileNet/ResNet) for better accuracy in less training time.

---
*Built as a course project for Artificial Intelligence (6th semester, BSCS) at the University of Central Punjab, under the supervision of Prof. Tajjamul.*